# CGI 静止数据质量审计

粒度：设备 SN × GPS 时间 × 协议。设备 6094510；用户确认 2026-08-26 15:08:55.100 至 18:39:57.900 全程未挪动。原始数据受现有存储策略保护；不将本次假设推广到未来或其他设备。

原始快照不随源码发布。运行前将一致性备份放在 `tmp/stationary-analysis.sqlite`，复制到 `tmp/filter-replay.sqlite` 后做回算。不要直接修改生产库。

In [ ]:
from pathlib import Path
import sys, json
root = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(root))
from analysis.profile_stationary import profile
from analysis.validate_filter import validate


## 分层统计
先剔除无定位/初始化状态估计位置中位数，不将 (0,0) 算成远距离漂移。其他通道保留原始分布以识别问题。此单设备脚本显式选择 SN，不能混合不同设备拟合。

In [ ]:
raw_profile = profile(root / "tmp/stationary-analysis.sqlite", device="6094510")
raw_profile["rows"], raw_profile["stats"]["speed"], raw_profile["stats"]["radius_m"]

## 已执行的验收结果
以下为执行 `analysis/validate_filter.py` 后保存的结果摘要；不是模拟数据。完整结果见 evidence/stationary-filter-validation.json，重跑代码见后续单元。

```json
{
  "raw_samples": 114008,
  "anomaly_samples": 7384,
  "raw_sha256": "10ebbdd33fead7ec93079e8a46d1fe02f182df99f23c148c15b292d221c211a5",
  "raw_unchanged": true,
  "summary": {
    "first_t": 1787728135.1,
    "last_t": 1787740797.9,
    "distance_km": 0.0,
    "moving_s": 0,
    "covered_s": 10969.409997701645,
    "max_kmh": 1.08,
    "fixed_pct": 0.0,
    "valid_pct": 96.36429022524734,
    "gap_count": 16,
    "fix_counts": {
      "6": 113240,
      "0": 768
    },
    "sample_hz": 9.003301007540415
  },
  "reason_counts": [
    {
      "code": "invalid_navigation",
      "label": "导航初始化 / 定位无效",
      "category": "unavailable",
      "count": 768
    },
    {
      "code": "heading_unavailable",
      "label": "定向未就绪",
      "category": "unavailable",
      "count": 114008
    },
    {
      "code": "course_unavailable",
      "label": "静止或低速航迹角不可用",
      "category": "unavailable",
      "count": 114008
    },
    {
      "code": "stationary_position",
      "label": "静止位置离群",
      "category": "anomaly",
      "count": 3377
    },
    {
      "code": "stationary_altitude",
      "label": "静止高程离群",
      "category": "anomaly",
      "count": 2549
    },
    {
      "code": "stationary_velocity",
      "label": "静止水平速度异常",
      "category": "anomaly",
      "count": 2746
    },
    {
      "code": "stationary_vertical",
      "label": "静止垂向速度异常",
      "category": "anomaly",
      "count": 1700
    },
    {
      "code": "stationary_gyro",
      "label": "静止角速度离群",
      "category": "anomaly",
      "count": 270
    },
    {
      "code": "stationary_accel",
      "label": "静止比力离群",
      "category": "anomaly",
      "count": 124
    },
    {
      "code": "stationary_attitude",
      "label": "静止姿态离群",
      "category": "anomaly",
      "count": 116
    }
  ]
}
```

In [ ]:
result = validate(root / "tmp/stationary-analysis.sqlite", root / "tmp/filter-replay.sqlite")
result["quality"]["reasons"], result["filtered_ranges"]

## 判据与限制
中位数与 MAD 抑制极端值对参考的污染。6×1.4826×MAD 和通道噪声下限为本项目保守工程起始值，不是法定标准。只遮蔽命中字段；未初始化航向与静止航迹角单独归类为不可用，不能因此删除整条 IMU 报文。数值离群也可能来自实际碰触/振动，应在隔离台账复核。

参考：[NIST Median Absolute Deviation](https://www.itl.nist.gov/div898/software/dataplot/refman2/auxillar/mad.htm)、CHC CGI-430 用户手册 V2.4.4。具体冻结参数：deploy/stationary-6094510-20260826.json。